In [1]:
import torch

# Создание разных тензоров
a = torch.tensor([1.0, 2.0, 3.0])  # 1D-тензор из списка
b = torch.zeros((2, 3), dtype=torch.int64)  # матрица 2×3 из нулей (64-битные целые)
c = torch.rand(3, 3)  # случайный тензор 3×3, равномерное распределение


print(a, a.dtype)  # tensor([1., 2., 3.]) torch.float32

print(b, b.dtype)  # tensor([[0, 0, 0],
#         [0, 0, 0]]) torch.int64

print(c)  # тензор из случайных чисел размером 3×3

tensor([1., 2., 3.]) torch.float32
tensor([[0, 0, 0],
        [0, 0, 0]]) torch.int64
tensor([[0.9551, 0.5032, 0.2350],
        [0.6981, 0.1275, 0.3077],
        [0.4780, 0.8228, 0.9266]])


In [ ]:
import torch

x = torch.tensor([1.0, -2.0, 3.0])
y = torch.exp(x)  # элемент-wise экспонента
z = x + y  # сложение поэлементно
print(z)

In [5]:
if torch.cuda.is_available():
    print("CUDA доступна. Перемещаем тензор на GPU.")
    a = a.to("cuda")  # переместить на CUDA-устройство
else:
    print("CUDA недоступна. Используем CPU.")
    a = a.to("cpu")  # переместить на CPU

CUDA недоступна. Используем CPU.


In [7]:
import torch

x = torch.ones(
    2, 2, requires_grad=True
)  # создаём тензор, указывая, что нужны градиенты
print(f"x: {x}")
y = x * 2
print(f"y: {y}")
z = y.mean()  # свёртка: усредняем все элементы
print(f"z: {z}")
z.backward()  # выполняем обратный проход (backpropagation)
print(f"x.grad: {x.grad}")  # смотрим градиент dz/dx

x: tensor([[1., 1.],
        [1., 1.]], requires_grad=True)
y: tensor([[2., 2.],
        [2., 2.]], grad_fn=<MulBackward0>)
z: 2.0
x.grad: tensor([[0.5000, 0.5000],
        [0.5000, 0.5000]])


In [8]:
from torch import nn


class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Создаём линейный слой: принимает 10 входов, выдаёт 5 выходов
        self.fc1 = nn.Linear(in_features=10, out_features=5)

In [10]:
from torch import nn


class SimpleNN(nn.Module):
    def __init__(self):
        # Базовая инициализация nn.Module
        # Ваш код здесь
        super().__init__()
        # Линейный слой: принимает тензор размера [..., 10], выдаёт [..., 5]
        # Ваш код здесь
        self.fc1 = nn.Linear(in_features=10, out_features=5)
        # Функция активации ReLU
        # Ваш код здесь
        self.relu = nn.ReLU()
        # Линейный слой: принимает [..., 5], выдаёт [..., 1]
        # Ваш код здесь
        self.fc2 = nn.Linear(in_features=5, out_features=1)

    def forward(self, x):
        # x - тензор размерности [batch_size, 10]
        x = self.fc1(x)  # теперь x → [batch_size, 5]
        # Примените ReLU
        x = self.relu(x)
        # Линейный слой x → [batch_size, 1]
        x = self.fc2(x)
        return x  # возвращаем тензор прогнозов


# Пример использования:
model = SimpleNN()
print(model)


# Можно сразу посмотреть, сколько параметров зарегистрировано:
total_params = sum(p.numel() for p in model.parameters())
print(f"Всего параметров: {total_params}")

SimpleNN(
  (fc1): Linear(in_features=10, out_features=5, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=5, out_features=1, bias=True)
)
Всего параметров: 61


In [13]:
from torch.utils.data import DataLoader, Dataset


class MyDataset(Dataset):
    def __init__(self, data, targets):
        self.data = data
        self.targets = targets

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.targets[idx]


# Создадим датасет из списков чисел
dataset = MyDataset(data=list(range(10)), targets=[2 * x for x in range(10)])

print(f"Длина датасета: {len(dataset)}")
print(dataset)

dataloader = DataLoader(dataset, batch_size=3, shuffle=True, num_workers=0)
for batch_data, batch_target in dataloader:
    print("Batch number {1}:")
    print(batch_data, batch_target)
    # break  # посмотрим только первый батч

Длина датасета: 10
Batch number {1}:
tensor([1, 5, 7]) tensor([ 2, 10, 14])
Batch number {1}:
tensor([4, 2, 9]) tensor([ 8,  4, 18])
Batch number {1}:
tensor([6, 3, 8]) tensor([12,  6, 16])
Batch number {1}:
tensor([0]) tensor([0])


In [ ]:
from torch import nn

# 1. Определяем подкласс nn.Module


class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Линейный слой: 5 нейронов на скрытом слое, принимающий 5 входов
        self.fc1 = nn.Linear(in_features=5, out_features=5)
        # ReLU-активация
        self.relu = nn.ReLU()
        # Выходной линейный слой: из 5 нейронов в 1 нейрон
        self.fc2 = nn.Linear(in_features=5, out_features=1)

    def forward(self, x):
        # x: [batch_size, 5]
        x = self.fc1(x)  # → [batch_size, 5]
        x = self.relu(x)  # → [batch_size, 5]
        x = self.fc2(x)  # → [batch_size, 1]
        return x


# 2. Создаём модель и оптимизатор
model = SimpleNN()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()


# 3. Генерируем случайные входные данные и метки
# Ваш код здесь
torch.manual_seed(0)
X = torch.randn(3, 5)  # случайный вход [batch_size=3, features=5]
y = torch.randn(3, 1)  # случайный таргет [batch_size=3, 1]


# 4. Сохраняем веса первого слоя до обучения
weights_before = model.fc1.weight.clone()


# 5. Прямой проход и лосс
outputs = model(X)  # Ваш код здесь
loss = criterion(outputs, y)  # Ваш код здесь
optimize = optimizer.zero_grad()  # обнуляем градиенты
loss.backward()  # вычисляем градиенты
optimizer.step()  # обновляем веса


# 6. Шаг обучения
# Ваш код здесь
print(f"Весовой тензор fc1.weight до обучения (кусочек):\n{weights_before[:2, :3]}")
print(
    f"Весовой тензор fc1.weight после обучения (кусочек):\n{model.fc1.weight[:2, :3]}"
)

# 7. Сохраняем веса первого слоя после обучения
# Ваш код здесь
weights_after = model.fc1.weight.clone()


# 8. Выводим кусочек весов до и после, а также значение loss

print("Весовой тензор fc1.weight до обучения (кусочек):")
print(weights_before[:2, :3])
print("\nВесовой тензор fc1.weight после обучения (кусочек):")
print(weights_after[:2, :3])
print(f"\nЗначение лосса: {loss.item():.4f}")

Весовой тензор fc1.weight до обучения (кусочек):
tensor([[-0.4441,  0.2819,  0.2000],
        [ 0.0467,  0.4156,  0.3494]], grad_fn=<SliceBackward0>)
Весовой тензор fc1.weight после обучения (кусочек):
tensor([[-0.4483,  0.2831,  0.2026],
        [ 0.0467,  0.4156,  0.3494]], grad_fn=<SliceBackward0>)
Весовой тензор fc1.weight до обучения (кусочек):
tensor([[-0.4441,  0.2819,  0.2000],
        [ 0.0467,  0.4156,  0.3494]], grad_fn=<SliceBackward0>)

Весовой тензор fc1.weight после обучения (кусочек):
tensor([[-0.4483,  0.2831,  0.2026],
        [ 0.0467,  0.4156,  0.3494]], grad_fn=<SliceBackward0>)

Значение лосса: 1.1573
